In [16]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

In [31]:
data = pd.read_csv('train.csv')
data.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:1000].T #flip it so each column is an example instead of each row.
Y_dev = data_dev[0]
X_dev = data_dev[1:n]
X_dev = X_dev / 255.

data_train = data[1000:m].T 
Y_train = data_train[0]
X_train = data_train[1:n]
X_train = X_train / 255.
_,m_train = X_train.shape

In [33]:
#initial param

def init_params():
    W1 = np.random.rand(10, 784) - 0.5 #rand dist. between 0 and 1
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1,W2, b2

#forward prompagation
def ReLU(Z):
    #ReLU 
    # x if x > 0
    # 0 if x <= 0
    return np.maximum(Z, 0) #goes throught all elements in Z, if one of them is greater than 0 returns Z, if not returns 0

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z)) #preserves amount of columns, but collapses rows to just 1, to have output of the range needed
    return A 

def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    #dot product between weight and input layer + bias
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

#Backward prompagation
def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def deriv_ReLU(Z):
    return Z > 0 # derivative of the slope is 0 or 1, and same with booleans

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * deriv_ReLU(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

In [34]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    print(predictions, Y)
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 10 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [35]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 500)

Iteration:  0
[7 7 7 ... 7 1 1] [7 4 8 ... 9 8 2]
0.09580487804878049
Iteration:  10
[7 7 7 ... 7 8 1] [7 4 8 ... 9 8 2]
0.18360975609756097
Iteration:  20
[7 7 7 ... 7 8 1] [7 4 8 ... 9 8 2]
0.2176341463414634
Iteration:  30
[7 7 7 ... 7 8 2] [7 4 8 ... 9 8 2]
0.27263414634146343
Iteration:  40
[7 7 9 ... 7 8 2] [7 4 8 ... 9 8 2]
0.30421951219512194
Iteration:  50
[7 7 9 ... 8 8 2] [7 4 8 ... 9 8 2]
0.32753658536585367
Iteration:  60
[7 7 9 ... 8 8 2] [7 4 8 ... 9 8 2]
0.36292682926829267
Iteration:  70
[7 9 9 ... 9 8 0] [7 4 8 ... 9 8 2]
0.41392682926829266
Iteration:  80
[7 9 8 ... 9 8 0] [7 4 8 ... 9 8 2]
0.45678048780487807
Iteration:  90
[7 9 8 ... 9 8 0] [7 4 8 ... 9 8 2]
0.48948780487804877
Iteration:  100
[7 9 8 ... 9 8 0] [7 4 8 ... 9 8 2]
0.5157560975609756
Iteration:  110
[9 9 8 ... 9 8 2] [7 4 8 ... 9 8 2]
0.5383902439024391
Iteration:  120
[9 9 8 ... 9 8 2] [7 4 8 ... 9 8 2]
0.5608048780487805
Iteration:  130
[9 9 8 ... 9 8 2] [7 4 8 ... 9 8 2]
0.5799756097560975
Iteratio